# 实战案例 4：Sim2Real — Pendulum 域随机化

**真实场景**：单关节机械臂/倒立摆力矩控制，在仿真中训练后在真机部署。真机摩擦、质量、传感器噪声与仿真不一致。

**本 Notebook 用 `Pendulum-v1` 模拟 Sim2Real**：
- **训练环境**：每 episode 随机重力/扭矩噪声（域随机化）
- **测试环境**：固定「真实」参数（与训练分布不同）
- 算法：Stable-Baselines3 SAC（连续控制）

`pip install gymnasium stable-baselines3 matplotlib numpy`

In [ ]:
!pip install -q gymnasium stable-baselines3 matplotlib numpy

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from gymnasium import Wrapper
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy

SEED = 42

## 1. 域随机化 Wrapper（模拟「仿真」）

In [ ]:
class DomainRandomizePendulum(Wrapper):
    """每 reset 随机重力缩放与动作噪声，模拟 sim 参数分布。"""
    def __init__(self, env, g_range=(0.7, 1.3), act_noise=0.1):
        super().__init__(env)
        self.g_range = g_range
        self.act_noise = act_noise
        self._g_scale = 1.0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._g_scale = np.random.uniform(*self.g_range)
        # Pendulum 内部 gravity 在 unwrapped
        if hasattr(self.env.unwrapped, "g"):
            base_g = 10.0
            self.env.unwrapped.g = base_g * self._g_scale
        return obs, info

    def step(self, action):
        if self.act_noise > 0:
            action = np.clip(action + np.random.randn(*np.shape(action)) * self.act_noise, -2, 2)
        return self.env.step(action)

def make_sim_env(seed=0):
    e = gym.make("Pendulum-v1")
    e = DomainRandomizePendulum(e, g_range=(0.6, 1.4), act_noise=0.15)
    e.reset(seed=seed)
    return e

def make_real_env(seed=0):
    """固定 g=12（偏离训练分布），模拟真机。"""
    e = gym.make("Pendulum-v1")
    e.reset(seed=seed)
    e.unwrapped.g = 12.0
    return e

## 2. 在随机化仿真中训练 SAC

In [ ]:
train_env = make_sim_env(SEED)
model = SAC("MlpPolicy", train_env, verbose=0, seed=SEED, learning_rate=3e-4)
model.learn(total_timesteps=30_000)  # 演示用；正式可 200k+
train_env.close()
print("SAC 训练完成（域随机化环境）")

## 3. 分别在 Sim / Real 环境评估

In [ ]:
def eval_env_factory(randomize, seed, n=10):
    rets = []
    for i in range(n):
        env = make_sim_env(seed + i) if randomize else make_real_env(seed + i)
        obs, _ = env.reset()
        done, ret = False, 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(action)
            ret += r; done = term or trunc
        rets.append(ret); env.close()
    return np.mean(rets), np.std(rets)

sim_m, sim_s = eval_env_factory(True, 100)
real_m, real_s = eval_env_factory(False, 200)
print(f"评估（域随机 sim）: {sim_m:.1f} ± {sim_s:.1f}")
print(f"评估（固定 real g=12）: {real_m:.1f} ± {real_s:.1f}")
print(f"Sim2Real gap: {sim_m - real_m:.1f}")

## 4. 无域随机化对照（反面教材）

In [ ]:
plain_env = gym.make("Pendulum-v1")
plain_model = SAC("MlpPolicy", plain_env, verbose=0, seed=SEED)
plain_model.learn(total_timesteps=30_000)
plain_env.close()

def eval_with_model(m, real=True, n=10):
    rets = []
    for i in range(n):
        env = make_real_env(300 + i) if real else make_sim_env(300 + i)
        obs, _ = env.reset()
        done, ret = False, 0
        while not done:
            action, _ = m.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(action)
            ret += r; done = term or trunc
        rets.append(ret); env.close()
    return np.mean(rets)

plain_sim = eval_with_model(plain_model, real=False)
plain_real = eval_with_model(plain_model, real=True)
print(f"无 DR - sim: {plain_sim:.1f}, real: {plain_real:.1f}, gap: {plain_sim-plain_real:.1f}")
print(f"有 DR - sim: {sim_m:.1f}, real: {real_m:.1f}, gap: {sim_m-real_m:.1f}")

In [ ]:
labels = ["DR+SAC\n(sim)", "DR+SAC\n(real)", "No DR\n(real)"]
vals = [sim_m, real_m, plain_real]
plt.bar(labels, vals, color=["steelblue", "coral", "gray"])
plt.ylabel("Mean return"); plt.title("域随机化对 Sim2Real 的影响")
plt.tight_layout(); plt.show()

## 5. 落地思考

| 方案 | 优势 | 局限 |
|------|------|------|
| 域随机化 | 无需真机数据、易并行 | 随机范围需调，过大难学 |
| 系统辨识 | 缩小 gap | 需标定实验 |
| IL+RL | 冷启动快 | 模仿数据质量依赖 |
| 安全护栏 | 可上线 | 可能限制最优性能 |

真机部署：**限速 → 硬件在环 → 影子模式 → 急停**。